# Binary Image Classification with a CNN — Cats vs Dogs
**Framework:** TensorFlow / Keras
**Dataset:** `cats_vs_dogs` via `tensorflow_datasets` (23,262 usable images, 2 classes)

This notebook builds, trains, and evaluates a Convolutional Neural Network for a two-class
image classification problem. It covers: data loading, preprocessing & augmentation,
model architecture, training, evaluation, and visualization of results.

> Runs on Google Colab (recommended: enable GPU via Runtime → Change runtime type → T4 GPU).


In [ ]:
# 1. Imports
import tensorflow as tf
from tensorflow.keras import layers, models
import tensorflow_datasets as tfds
import matplotlib.pyplot as plt
import numpy as np

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))


In [ ]:
# Install the dataset package if needed (Colab/Jupyter)
%pip install -q -U tensorflow-datasets


## 2. Load the Dataset
`tensorflow_datasets` ships a ready-to-use `cats_vs_dogs` dataset. It has only a `train`
split, so we manually carve out train/validation/test slices (80/10/10).


## 3. Preprocess & Build Input Pipelines

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers
def preprocess(image, label):
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = tf.cast(image, tf.float32) / 255.0   # normalize to [0, 1]
    return image, label

# Light data augmentation applied only to the training set
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

def prepare(ds, shuffle=False, augment=False):
    ds = ds.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(1000)
    ds = ds.batch(BATCH_SIZE)
    if augment:
        ds = ds.map(lambda x, y: (data_augmentation(x, training=True), y),
                    num_parallel_calls=tf.data.AUTOTUNE)
    return ds.prefetch(tf.data.AUTOTUNE)

train_ds = prepare(raw_train, shuffle=True, augment=True)
val_ds   = prepare(raw_val)
test_ds  = prepare(raw_test)


In [ ]:
# Sanity-check: visualize a batch
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 10))
for images, labels in train_ds.take(1):
    for i in range(9):
        plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy())
        plt.title(class_names[labels[i].numpy()])
        plt.axis('off')
plt.tight_layout()
plt.show()


## 4. Build the CNN
A compact CNN with three convolution/pooling blocks, batch normalization, and dropout
for regularization. Output is a single sigmoid unit for binary classification.


In [ ]:
model = models.Sequential([
    layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),

    layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')   # binary output
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()


## 5. Train

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True, monitor='val_loss'),
    tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=3, monitor='val_loss'),
]

EPOCHS = 20

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks
)


## 6. Evaluate on the Held-Out Test Set

In [ ]:
test_loss, test_acc = model.evaluate(test_ds)
print(f"Test accuracy: {test_acc:.4f}")
print(f"Test loss: {test_loss:.4f}")


## 7. Visualize Training Curves

In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']
epochs_range = range(len(acc))

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Train Accuracy')
plt.plot(epochs_range, val_acc, label='Val Accuracy')
plt.legend(loc='lower right')
plt.title('Accuracy')

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Train Loss')
plt.plot(epochs_range, val_loss, label='Val Loss')
plt.legend(loc='upper right')
plt.title('Loss')

plt.tight_layout()
plt.show()


## 8. Confusion Matrix & Classification Report

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

y_true = []
y_pred = []

for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    y_pred.extend((preds.flatten() > 0.5).astype(int))
    y_true.extend(labels.numpy())

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

print(classification_report(y_true, y_pred, target_names=class_names))


## 9. Predict on a Single New Image (optional)
Upload any cat/dog image and run it through the trained model.


In [ ]:
def predict_image(img_path):
    img = tf.keras.utils.load_img(img_path, target_size=(IMG_SIZE, IMG_SIZE))
    img_array = tf.keras.utils.img_to_array(img) / 255.0
    img_array = tf.expand_dims(img_array, 0)
    pred = model.predict(img_array, verbose=0)[0][0]
    label = class_names[1] if pred > 0.5 else class_names[0]
    confidence = pred if pred > 0.5 else 1 - pred
    print(f"Prediction: {label} ({confidence:.2%} confidence)")

# Example (uncomment and set your own path in Colab after uploading a file):
# from google.colab import files
# uploaded = files.upload()
# predict_image(list(uploaded.keys())[0])


## 10. Save the Model

In [ ]:
model.save('cats_vs_dogs_cnn.keras')
print("Model saved as cats_vs_dogs_cnn.keras")
